In [18]:
!pip install -q bitsandbytes

# 01 · Data Labeling (Human-in-the-loop)
Giai đoạn 1: auto-label bằng model **Qwen2.5-VL-7B-Instruct** (full bf16 trên GPU **L4 24GB**, tải từ HuggingFace về local) sinh nhãn draft, rồi duyệt tay bằng Gradio để chốt thành **ground truth**.

> Đổi GPU trong Colab: **Runtime → Change runtime type → L4 GPU**. Nếu chỉ có T4 16GB, thêm cờ `--load_4bit` vào 2 cell mục 4 để nén NF4.

**Mọi output (draft JSON, ảnh raw, model) lưu thẳng vào Google Drive** để không mất khi Colab ngắt.

## 1. Mount Google Drive + thiết lập đường dẫn

In [19]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Gốc dự án trên Drive — trỏ thẳng vào thư mục Data
PROJECT_DRIVE = Path('/content/drive/MyDrive/cccd_project/Data')

# Các thư mục chứa dữ liệu và model
FRONT_DIR  = PROJECT_DRIVE / 'Front'
BACK_DIR   = PROJECT_DRIVE / 'Back'
DRAFT_DIR  = PROJECT_DRIVE / 'draft'    # nhãn draft (JSONL)
MODELS_DIR = PROJECT_DRIVE / 'models'   # model tải về local

# Model DUY NHẤT cho toàn bộ luồng label: Qwen2.5-VL-7B (full bf16 trên L4 24GB).
QWEN7B_LOCAL = MODELS_DIR / 'Qwen2.5-VL-7B-Instruct'

for d in (FRONT_DIR, BACK_DIR, DRAFT_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('FRONT  :', FRONT_DIR)
print('BACK   :', BACK_DIR)
print('DRAFT  :', DRAFT_DIR)
print('MODEL 7B:', QWEN7B_LOCAL)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FRONT  : /content/drive/MyDrive/cccd_project/Data/Front
BACK   : /content/drive/MyDrive/cccd_project/Data/Back
DRAFT  : /content/drive/MyDrive/cccd_project/Data/draft
MODEL 7B: /content/drive/MyDrive/cccd_project/Data/models/Qwen2.5-VL-7B-Instruct


## 2. Lấy source code + cài thư viện

In [20]:
# Dọn dẹp thư mục làm việc cũ (nếu có) và tạo mới
!rm -rf /content/cccd
!mkdir -p /content/cccd

# Copy TOÀN BỘ nội dung của label_CCCD (bao gồm src, notebooks...) vào Colab
!cp -r /content/drive/MyDrive/cccd_project/Data/label_CCCD/* /content/cccd/ 2>/dev/null || echo 'Cảnh báo: Không tìm thấy source code!'

%cd /content/cccd
!pip -q install 'transformers>=4.49.0' qwen-vl-utils accelerate Pillow tqdm gradio huggingface_hub

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
/content/cccd


## 3. Tải Qwen2.5-VL-7B-Instruct từ HuggingFace về LOCAL (Drive)
Tải 1 lần, lần sau bỏ qua nếu đã có. Đây là model DUY NHẤT cho cả Front lẫn Back (full bf16 trên L4 24GB).

In [ ]:
from huggingface_hub import snapshot_download

if not (QWEN7B_LOCAL / 'config.json').exists():
    snapshot_download(repo_id='Qwen/Qwen2.5-VL-7B-Instruct',
                      local_dir=str(QWEN7B_LOCAL),
                      ignore_patterns=['*.pth', 'original/*'])
    print('✓ Đã tải 7B về', QWEN7B_LOCAL)
else:
    print('✓ Model đã có sẵn local:', QWEN7B_LOCAL)

✓ Model đã có sẵn local: /content/drive/MyDrive/cccd_project/Data/models/Qwen2.5-VL-7B-Instruct


## 4. Auto-label (sinh nhãn draft) — Qwen2.5-VL-7B (full bf16, L4 24GB), prompt động theo tên file front/back

Một lượt 7B duy nhất cho cả Front và Back (đã bỏ chiến lược 3B + retry tầng-2). Prompt đã có quy tắc đọc văn bản xuống dòng (nối dòng + thêm phẩy; gặp gạch nối cuối dòng thì nối liền, không thêm phẩy).

In [ ]:
# Chạy cho mặt trước — Qwen2.5-VL-7B FULL bf16 trên L4 24GB (không nén, chất lượng cao nhất).
# --max_pixels 1605632 (~1.6MP, =2048*28*28): nét chữ tốt mà vẫn an toàn VRAM.
# (L4 còn dư có thể bỏ --max_pixels để dùng mặc định ~2MP; T4 16GB thì thêm --load_4bit.)
!python -m src.data_pipeline.auto_label \
    --input_dir '{FRONT_DIR}' \
    --result_dir '{DRAFT_DIR}' \
    --model_name '{QWEN7B_LOCAL}' \
    --max_pixels 1605632

/usr/bin/python3: Error while finding module specification for 'src.data_pipeline.auto_label' (ModuleNotFoundError: No module named 'src')


In [ ]:
# Chạy cho mặt sau — Qwen2.5-VL-7B FULL bf16 trên L4 24GB (không nén).
!python -m src.data_pipeline.auto_label \
    --input_dir '{BACK_DIR}' \
    --result_dir '{DRAFT_DIR}' \
    --model_name '{QWEN7B_LOCAL}' \
    --max_pixels 1605632

2026-08-02 06:40:11,574 [INFO] ============================================================
2026-08-02 06:40:11,574 [INFO] Model     : /content/drive/MyDrive/cccd_project/Data/models/Qwen2.5-VL-7B-Instruct
2026-08-02 06:40:11,574 [INFO] Input     : /content/drive/MyDrive/cccd_project/Data/Back
2026-08-02 06:40:11,574 [INFO] Output    : /content/drive/MyDrive/cccd_project/Data/draft/Back_draft.jsonl
2026-08-02 06:40:11,574 [INFO] ============================================================
2026-08-02 06:40:11,574 [INFO] Loading: /content/drive/MyDrive/cccd_project/Data/models/Qwen2.5-VL-7B-Instruct (min_pixels=200704, max_pixels=1605632, 4bit=False)
2026-08-02 06:40:23,971 [INFO] NumExpr defaulting to 12 threads.
Loading weights: 100% 729/729 [05:00<00:00,  2.43it/s]
2026-08-02 06:45:43,384 [INFO] ✓ Model loaded: /content/drive/MyDrive/cccd_project/Data/models/Qwen2.5-VL-7B-Instruct
2026-08-02 06:45:43,552 [INFO] [Back] Tổng ảnh: 1602
2026-08-02 06:45:43,553 [INFO] [Back] Cần label: 160

## 5. Duyệt & sửa nhãn bằng Gradio (chạy LOCAL trên máy)

Chạy ngay trên máy local: ảnh nạp từ `data/Front` và `data/Back`, file JSONL
(`Front_draft.jsonl` / `Back_draft.jsonl`) đặt ở thư mục gốc `label_CCCD`.
Mở trình duyệt tại http://127.0.0.1:7860.

- Trường `ngay_het_han` đã đổi tên hiển thị/lưu thành **`co_gia_tri_den`**.
- Ô **Jump to index**: gõ số (vd `017`) để nhảy thẳng tới ảnh
  `cccd_front_17` hoặc `cccd_back_17` (tùy file draft đang nạp); báo rõ tên file đã tới.
- **Tăng tốc duyệt 8k+ ảnh:**
  - **💾 Save & Next** giờ nhảy thẳng tới ảnh **CHƯA duyệt** kế tiếp (bỏ qua ảnh đã duyệt).
  - Tick **🔎 Chỉ duyệt ca CHƯA duyệt** → Prev/Skip chỉ chạy trong nhóm chưa duyệt.
  - UI tự mở ở ảnh chưa duyệt đầu tiên (tiện resume giữa chừng).

In [21]:
!pip -q install gradio Pillow

In [ ]:
# ── Duyệt nhãn trên GOOGLE COLAB bằng Gradio (chạy IN-PROCESS, hiện UI ngay trong notebook) ──
# Chạy trực tiếp trong kernel (KHÔNG dùng !python) để: UI render inline,
# in URL ngay lập tức, và mọi lỗi (vd thiếu file draft) hiện ra ngay.
if 'demo' in locals():
    try:
        demo.close()
    except:
        pass
import sys
from pathlib import Path

# Thư mục source code (label_CCCD) đã copy vào Colab ở mục 2 → để import package src.*
PROJECT_ROOT = Path("/content/drive/MyDrive/cccd_project/Data/label_CCCD/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_pipeline.label_tool import LabelStore, build_ui

# Dữ liệu (ảnh + draft) đọc thẳng từ Google Drive.
DATA_DIR  = Path("/content/drive/MyDrive/cccd_project/Data")
FRONT_DIR = DATA_DIR / "Front"
BACK_DIR  = DATA_DIR / "Back"
DRAFT_DIR = DATA_DIR / "draft"   # chứa Front_draft.jsonl và Back_draft.jsonl

# Đổi 'Back_draft.jsonl' <-> 'Front_draft.jsonl' tùy mặt muốn duyệt.
# Jump to index sẽ trả ảnh cccd_front_* hay cccd_back_* theo đúng file này.
DRAFT_FILE = DRAFT_DIR / "Front_draft.jsonl"
assert DRAFT_FILE.exists(), f"Không thấy file draft: {DRAFT_FILE}"

print("ROOT  :", PROJECT_ROOT)
print("DRAFT :", DRAFT_FILE)
print("FRONT :", FRONT_DIR)
print("BACK  :", BACK_DIR)

store = LabelStore(DRAFT_FILE, DATA_DIR, front_dir=FRONT_DIR, back_dir=BACK_DIR)
demo = build_ui(store)

# allowed_paths: BẮT BUỘC để Gradio được phép phục vụ ảnh nằm NGOÀI cwd
# (ảnh ở Drive, còn cwd là /content/cccd). Thiếu nó → ảnh + mọi ô bị "Error".
# prevent_thread_lock=True → cell chạy xong ngay, server chạy nền (không treo cell).
_, local_url, share_url = demo.launch(
    prevent_thread_lock=True,
    share=True,   # Colab không truy cập được 127.0.0.1 → cần link công khai để mở UI
    allowed_paths=[str(DATA_DIR), str(FRONT_DIR), str(BACK_DIR)],
)
print("→ Mở UI tại:", share_url or local_url)
# Muốn dừng server: demo.close()

ROOT  : /content/drive/MyDrive/cccd_project/Data/label_CCCD
DRAFT : /content/drive/MyDrive/cccd_project/Data/draft/Front_draft.jsonl
FRONT : /content/drive/MyDrive/cccd_project/Data/Front
BACK  : /content/drive/MyDrive/cccd_project/Data/Back
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://424951c05bcb6bb327.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


→ Mở UI tại: https://424951c05bcb6bb327.gradio.live


In [ ]:
# ── Duyệt nhãn trên GOOGLE COLAB bằng Gradio (chạy IN-PROCESS, hiện UI ngay trong notebook) ──
# Chạy trực tiếp trong kernel (KHÔNG dùng !python) để: UI render inline,
# in URL ngay lập tức, và mọi lỗi (vd thiếu file draft) hiện ra ngay.
import sys
from pathlib import Path

# Thư mục source code (label_CCCD) đã copy vào Colab ở mục 2 → để import package src.*
PROJECT_ROOT = Path("/content/cccd")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_pipeline.label_tool import LabelStore, build_ui

# Dữ liệu (ảnh + draft) đọc thẳng từ Google Drive.
DATA_DIR  = Path("/content/drive/MyDrive/cccd_project/Data")
FRONT_DIR = DATA_DIR / "Front"
BACK_DIR  = DATA_DIR / "Back"
DRAFT_DIR = DATA_DIR / "draft"   # chứa Front_draft.jsonl và Back_draft.jsonl

# Đổi 'Back_draft.jsonl' <-> 'Front_draft.jsonl' tùy mặt muốn duyệt.
# Jump to index sẽ trả ảnh cccd_front_* hay cccd_back_* theo đúng file này.
DRAFT_FILE = DRAFT_DIR / "Back_draft.jsonl"
assert DRAFT_FILE.exists(), f"Không thấy file draft: {DRAFT_FILE}"

print("ROOT  :", PROJECT_ROOT)
print("DRAFT :", DRAFT_FILE)
print("FRONT :", FRONT_DIR)
print("BACK  :", BACK_DIR)

store = LabelStore(DRAFT_FILE, DATA_DIR, front_dir=FRONT_DIR, back_dir=BACK_DIR)
demo = build_ui(store)

# allowed_paths: BẮT BUỘC để Gradio được phép phục vụ ảnh nằm NGOÀI cwd
# (ảnh ở Drive, còn cwd là /content/cccd). Thiếu nó → ảnh + mọi ô bị "Error".
# prevent_thread_lock=True → cell chạy xong ngay, server chạy nền (không treo cell).
_, local_url, share_url = demo.launch(
    prevent_thread_lock=True,
    share=True,   # Colab không truy cập được 127.0.0.1 → cần link công khai để mở UI
    allowed_paths=[str(DATA_DIR), str(FRONT_DIR), str(BACK_DIR)],
)
print("→ Mở UI tại:", share_url or local_url)
# Muốn dừng server: demo.close()

ROOT  : /content/cccd
DRAFT : /content/drive/MyDrive/cccd_project/Data/draft/Back_draft.jsonl
FRONT : /content/drive/MyDrive/cccd_project/Data/Front
BACK  : /content/drive/MyDrive/cccd_project/Data/Back
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://970f19a3aa5c1e8c96.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import json
import re
import shutil
from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive nếu chưa mount
if not Path("/content/drive").exists():
    drive.mount("/content/drive")

# 2. Cấu hình đường dẫn file trên Drive
DRAFT_FILE = Path("/content/drive/MyDrive/cccd_project/Data/draft/Back_draft.jsonl")
BACKUP_FILE = DRAFT_FILE.with_suffix(".jsonl.bak")
MIN_INDEX = 505  # Ngưỡng STT bắt đầu gán reviewed = True

def mark_reviewed_colab(file_path, min_idx=505):
    if not file_path.exists():
        print(f"❌ Lỗi: Không tìm thấy file tại đường dẫn:\n   {file_path}")
        return

    # Sao lưu file gốc trước khi ghi đè để an toàn
    shutil.copy2(file_path, BACKUP_FILE)
    print(f"🔒 Đã sao lưu file gốc sang: {BACKUP_FILE.name}")

    updated_count = 0
    total_count = 0
    new_records = []

    # Đọc và cập nhật từng dòng JSONL
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            total_count += 1
            record = json.loads(line)

            # Lấy tên file ảnh (vd: /content/.../cccd_back_505.png -> cccd_back_505.png)
            image_path = record.get("image", "")
            filename = Path(image_path).name

            # Trích xuất số thứ tự từ tên file
            match = re.search(r'(\d+)', filename)
            if match:
                img_num = int(match.group(1))
                # Nếu STT >= 505 thì cập nhật reviewed = True
                if img_num >= min_idx:
                    if "_meta" not in record or not isinstance(record["_meta"], dict):
                        record["_meta"] = {}

                    record["_meta"]["reviewed"] = True
                    updated_count += 1

            new_records.append(record)

    # Ghi đè lại dữ liệu đã cập nhật vào Back_draft.jsonl trên Drive
    with open(file_path, "w", encoding="utf-8") as f:
        for rec in new_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print("\n✅ HOÀN THÀNH CẬP NHẬT TRÊN DRIVE!")
    print(f"• Tổng số record trong file: {total_count}")
    print(f"• Số record (STT ≥ {min_idx}) đã đổi thành 'reviewed': True -> {updated_count}")

# Chạy cập nhật
mark_reviewed_colab(DRAFT_FILE, min_idx=MIN_INDEX)

🔒 Đã sao lưu file gốc sang: Back_draft.jsonl.bak

✅ HOÀN THÀNH CẬP NHẬT TRÊN DRIVE!
• Tổng số record trong file: 1602
• Số record (STT ≥ 505) đã đổi thành 'reviewed': True -> 1098
